In [ ]:
"""
NPY Image Visualization Tool
Visualize RGB images stored in .npy format.
"""

import numpy as np
import matplotlib.pyplot as plt


def visualize_npy_image(file_path: str) -> None:
    """
    Load and visualize an RGB image from a .npy file.

    Args:
        file_path: Path to the .npy file containing image data.
    """
    try:
        # Load data from the .npy file
        image_data = np.load(file_path)

        # Check array shape to ensure it's a valid RGB image
        # A typical RGB image array has shape (height, width, 3)
        if image_data.ndim != 3 or image_data.shape[2] != 3:
            print(f"Error: Data in '{file_path}' is not a valid RGB image format.")
            print(f"Array shape: {image_data.shape}, expected: (height, width, 3)")
            return

        # Display the image using matplotlib
        plt.figure(figsize=(10, 8))
        plt.imshow(image_data)
        plt.title(f"Visualization: {file_path}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"Error processing file: {e}")


# Example usage
npy_file = '../datasets/MMFi/E01/S01/A01/infra1/frame001.npy'
visualize_npy_image(npy_file)

In [ ]:
"""
C3D Motion Capture File Reader
Parse and extract 3D marker coordinates from .c3d files.
"""

import c3d
import numpy as np
from typing import Tuple, List, Optional


def read_c3d_data(file_path: str) -> Tuple[Optional[np.ndarray], List[str]]:
    """
    Read and parse a .c3d file, extracting 3D marker coordinate data.
    
    Args:
        file_path: Path to the .c3d file.
        
    Returns:
        Tuple of (marker_coordinates, marker_labels):
            - marker_coordinates: Array of shape (num_frames, num_markers, 3)
            - marker_labels: List of marker label names
    """
    try:
        with open(file_path, 'rb') as handle:
            reader = c3d.Reader(handle)

            # Print basic file information
            print("--- File Information ---")
            print(f"Number of markers: {reader.header.point_count}")
            print(f"First frame: {reader.header.first_frame}")
            print(f"Last frame: {reader.header.last_frame}")
            print(f"Total frames: {reader.frame_count}")
            print(f"Frame rate (Hz): {reader.header.frame_rate}")
            print("-" * 25)

            # Get marker labels
            point_labels = reader.point_labels
            if point_labels:
                print("\n--- Marker Labels ---")
                for i, label in enumerate(point_labels):
                    print(f"  {i}: {label.strip()}")
                print("-" * 25)
            else:
                print("No marker labels found in file.")

            # Extract all frame data
            print("\n--- Data Extraction ---")
            frames = list(reader.read_frames())

            if not frames:
                print("No data frames found in file.")
                return None, []

            first_frame_points = frames[0][1]
            num_points = first_frame_points.shape[0]
            print(f"Successfully read {len(frames)} frames.")
            print(f"Each frame contains {num_points} markers.")
            print(f"Data array shape: {first_frame_points.shape}")

            # Extract 3D coordinates into a NumPy array
            all_frames_xyz = np.zeros((len(frames), num_points, 3))
            for i, (frame_num, points, analog) in enumerate(frames):
                all_frames_xyz[i, :, :] = points[:, :3]

            print(f"\nConsolidated coordinates array shape: {all_frames_xyz.shape}")
            
            if point_labels:
                first_marker_name = point_labels[0].strip()
                first_marker_trajectory = all_frames_xyz[:, 0, :]
                print(f"'{first_marker_name}' trajectory shape: {first_marker_trajectory.shape}")

            return all_frames_xyz, list(point_labels)

    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None, []
    except Exception as e:
        print(f"Error reading/parsing file: {e}")
        return None, []


# Example usage
c3d_file = '../datasets/MMFi/E01/S01/A01/ground_truth.c3d'
trajectory_data, labels = read_c3d_data(c3d_file)

# Print first 5 frames of the first marker
if trajectory_data is not None and labels:
    print(f"\n--- Sample Data ---")
    label_name = labels[0].strip()
    print(f"Marker '{label_name}' coordinates for first 5 frames:")
    for i in range(min(5, trajectory_data.shape[0])):
        print(f"  Frame {i+1}: {trajectory_data[i, 0, :]}")